In [22]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [23]:
df = pd.read_csv(r'C:\Users\admin\Documents\My_projects\AQI_project\data\final_integrated_data.csv')
df = df.sort_values("Timestamp").reset_index(drop= True)
df.shape

(29222, 52)

In [24]:
y = df["PM2.5 (µg/m³)"]
y.shape

(29222,)

In [25]:
# Experiment #1 features - CAMS composition

cams_composition = ['co', 'aermr04', 'aermr05', 'aermr06', 'aermr09', 'aermr07', 'aermr10',
       'aermr08', 'no2', 'no', 'go3','so2']

X1 = df[cams_composition]
X1.shape

(29222, 12)

In [26]:
# Experiment #2 features - CAMS composition + meteorology

cams_meteorology = ['q','t','u10', 'v10', 'd2m', 't2m']

X2 = df[cams_composition + cams_meteorology]
X2.shape

(29222, 18)

In [27]:
# EXPERIMENT 1

from sklearn.model_selection import train_test_split
x1_train,x1_test,y_train,y_test = train_test_split(X1,y,test_size=0.20,shuffle= False)  # don't want to shuffle the data , want to test from the last 2 year data
x1_train.shape,x1_test.shape,y_train.shape,y_test.shape

((23377, 12), (5845, 12), (23377,), (5845,))

In [28]:
# EXPERIMENT 2

from sklearn.model_selection import train_test_split
x2_train,x2_test,y2_train,y2_test = train_test_split(X2,y,test_size=0.20,shuffle= False)
x2_train.shape,x2_test.shape,y2_train.shape,y2_test.shape

((23377, 18), (5845, 18), (23377,), (5845,))

In [29]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [30]:
# model creation

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor,GradientBoostingRegressor,AdaBoostRegressor)
from xgboost import XGBRegressor

In [32]:
models = {
    "Decision Tree" : DecisionTreeRegressor(random_state=42),
    "Random Forest" : RandomForestRegressor(random_state=42),
    "Gradient Boosting" :GradientBoostingRegressor(random_state=42),
    "AdaBoost" : AdaBoostRegressor(random_state=42),
    "XGBoost" : XGBRegressor(random_state=42)
}

In [33]:
results_exp1 = {}

for name, model in models.items():
    model.fit(x1_train, y_train)
    y_pred = model.predict(x1_test)

    results_exp1[name] = y_pred

In [34]:
results_exp1

{'Decision Tree': array([110.625, 207.75 , 108.   , ..., 295.125, 210.25 , 272.   ],
       shape=(5845,)),
 'Random Forest': array([127.64451292, 165.19349098, 205.3101875 , ..., 230.11369231,
        214.67477778, 266.76742381], shape=(5845,)),
 'Gradient Boosting': array([113.99418883, 137.68858456, 202.71699934, ..., 255.54185274,
        225.57951515, 261.41781894], shape=(5845,)),
 'AdaBoost': array([153.5426892 , 271.81523198, 316.99962789, ..., 429.1993267 ,
        319.07831525, 433.13295474], shape=(5845,)),
 'XGBoost': array([109.337326, 154.53867 , 183.53372 , ..., 247.4404  , 196.04274 ,
        257.83813 ], shape=(5845,), dtype=float32)}

In [35]:
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

In [36]:
evaluation_exp1 = []

for name, y_pred in results_exp1.items():
    evaluation_exp1.append({
        "Model": name,
        "R2": r2_score(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": mean_squared_error(y_test, y_pred) ** 0.5
    })

evaluation_exp1 = pd.DataFrame(evaluation_exp1)

evaluation_exp1

,Model,R2,MAE,RMSE
0,Decision Tree,0.154634,67.301269,105.408336
1,Random Forest,0.608984,48.482662,71.688548
2,Gradient Boosting,0.608351,47.872780,71.746541
3,AdaBoost,-0.849173,136.416847,155.898290
4,XGBoost,0.577705,49.017646,74.500769


In [37]:
results_exp2 = {}

for name, model in models.items():
    model.fit(x2_train, y2_train)
    y_pred = model.predict(x2_test)

    results_exp2[name] = y_pred

In [38]:
evaluation_exp2 = []

for name, y_pred in results_exp2.items():
    evaluation_exp2.append({
        "Model": name,
        "R2": r2_score(y2_test, y_pred),
        "MAE": mean_absolute_error(y2_test, y_pred),
        "RMSE": mean_squared_error(y2_test, y_pred) ** 0.5
    })

evaluation_exp2 = pd.DataFrame(evaluation_exp2)

evaluation_exp2

,Model,R2,MAE,RMSE
0,Decision Tree,0.257475,62.558415,98.788884
1,Random Forest,0.645448,46.149061,68.264162
2,Gradient Boosting,0.640197,45.652413,68.767782
3,AdaBoost,-0.772978,139.633544,152.652640
4,XGBoost,0.629806,46.411764,69.753717


In [39]:
comparison = evaluation_exp1.merge(
    evaluation_exp2,
    on="Model",
    suffixes=("_Exp1", "_Exp2")
)

comparison

,Model,R2_Exp1,MAE_Exp1,RMSE_Exp1,R2_Exp2,MAE_Exp2,RMSE_Exp2
0,Decision Tree,0.154634,67.301269,105.408336,0.257475,62.558415,98.788884
1,Random Forest,0.608984,48.482662,71.688548,0.645448,46.149061,68.264162
2,Gradient Boosting,0.608351,47.872780,71.746541,0.640197,45.652413,68.767782
3,AdaBoost,-0.849173,136.416847,155.898290,-0.772978,139.633544,152.652640
4,XGBoost,0.577705,49.017646,74.500769,0.629806,46.411764,69.753717
